In [12]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [13]:
seed = 42
shuffle = True
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [14]:
load_model = efficientnet_b0(
     weights=EfficientNet_B0_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

In [15]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [16]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]
#
df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

#### view data

In [17]:
(df_train.shape, df_val.shape, df_test.shape)

((7219, 5), (1805, 5), (1592, 4))

In [18]:
from utils.dataset import RGB2LUVTransform

transforms = Albu.Compose([
    RGB2LUVTransform(p=1.0),
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])
val_transform = Albu.Compose([
    RGB2LUVTransform(p=1.0),
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

In [19]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=val_transform)
test_dataset = PandasDataset(images_dir, df_test, transforms=val_transform)

In [20]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [21]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [22]:
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b0-luv.txt",
    path_to_save_model="models/b0-luv.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 301/301 [03:59<00:00,  1.26it/s]


VAL_LOSS     0.283
VAL_ACC      Mean: 50.159 | Std: 1.177 | 95% CI: [48.310, 52.133]
VAL_KAPPA    Mean: 0.787 | Std: 0.011 | 95% CI: [0.768, 0.804]
VAL_F1       Mean: 0.458 | Std: 0.012 | 95% CI: [0.437, 0.478]
VAL_RECALL   Mean: 0.459 | Std: 0.012 | 95% CI: [0.439, 0.479]
VAL_PRECISION Mean: 0.531 | Std: 0.013 | 95% CI: [0.510, 0.551]
Salvando o melhor modelo... 0.0 -> 0.7867822635766392
Epoch 2/50



100%|██████████| 301/301 [03:57<00:00,  1.27it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.309
VAL_ACC      Mean: 51.834 | Std: 1.211 | 95% CI: [49.751, 53.850]
VAL_KAPPA    Mean: 0.753 | Std: 0.013 | 95% CI: [0.732, 0.774]
VAL_F1       Mean: 0.432 | Std: 0.012 | 95% CI: [0.411, 0.452]
VAL_RECALL   Mean: 0.431 | Std: 0.011 | 95% CI: [0.413, 0.449]
VAL_PRECISION Mean: 0.510 | Std: 0.014 | 95% CI: [0.486, 0.533]
Epoch 3/50



...

VAL_LOSS     0.372
VAL_ACC      Mean: 54.551 | Std: 1.197 | 95% CI: [52.632, 56.620]
VAL_KAPPA    Mean: 0.762 | Std: 0.013 | 95% CI: [0.740, 0.783]
VAL_F1       Mean: 0.453 | Std: 0.012 | 95% CI: [0.433, 0.473]
VAL_RECALL   Mean: 0.450 | Std: 0.011 | 95% CI: [0.433, 0.469]
VAL_PRECISION Mean: 0.515 | Std: 0.014 | 95% CI: [0.493, 0.538]
Epoch 5/50



100%|██████████| 301/301 [04:10<00:00,  1.20it/s]


VAL_LOSS     0.475
VAL_ACC      Mean: 46.508 | Std: 1.156 | 95% CI: [44.598, 48.476]
VAL_KAPPA    Mean: 0.674 | Std: 0.015 | 95% CI: [0.648, 0.699]
VAL_F1       Mean: 0.390 | Std: 0.011 | 95% CI: [0.372, 0.410]
VAL_RECALL   Mean: 0.396 | Std: 0.011 | 95% CI: [0.379, 0.415]
VAL_PRECISION Mean: 0.496 | Std: 0.016 | 95% CI: [0.470, 0.522]
Epoch 6/50



100%|██████████| 301/301 [04:11<00:00,  1.20it/s]


VAL_LOSS     0.392
VAL_ACC      Mean: 56.242 | Std: 1.193 | 95% CI: [54.349, 58.285]
VAL_KAPPA    Mean: 0.782 | Std: 0.013 | 95% CI: [0.761, 0.802]
VAL_F1       Mean: 0.505 | Std: 0.012 | 95% CI: [0.485, 0.525]
VAL_RECALL   Mean: 0.502 | Std: 0.012 | 95% CI: [0.483, 0.522]
VAL_PRECISION Mean: 0.557 | Std: 0.012 | 95% CI: [0.537, 0.578]

Early stopping at epoch 6. No improvement for 5 epochs.
Best epoch: 1 with kappa: 0.7868


# tests

In [23]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b0-luv.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

100%|██████████| 266/266 [03:36<00:00,  1.23it/s]


VAL_ACC      Mean: 49.512 | Std: 1.269 | 95% CI: [47.362, 51.573]
VAL_KAPPA    Mean: 0.781 | Std: 0.013 | 95% CI: [0.760, 0.802]
VAL_F1       Mean: 0.448 | Std: 0.013 | 95% CI: [0.426, 0.470]
VAL_RECALL   Mean: 0.453 | Std: 0.013 | 95% CI: [0.432, 0.475]
VAL_PRECISION Mean: 0.533 | Std: 0.013 | 95% CI: [0.511, 0.555]
